In [1]:
import pandas as pd
import numpy as np


# 원본 데이터 불러오기
data = pd.read_csv(
    "../risk_data/Sleep Health and Lifestyle Dataset.csv"
)

print("=== 데이터 크기 ===")
print(data.shape)

print("\n=== 분석 대상 Feature ===")
display(
    data[
        [
            "Sleep_Duration",
            "Daytime_Sleepiness",
            "Stress_Level",
            "Sleep_Efficiency"
        ]
    ].head()
)

=== 데이터 크기 ===
(30000, 39)

=== 분석 대상 Feature ===


,Sleep_Duration,Daytime_Sleepiness,Stress_Level,Sleep_Efficiency
0,5.66,5,7,80.4
1,6.93,1,4,93.6
2,7.18,2,4,97.1
3,7.77,2,4,90.7
4,7.23,2,4,99.0


In [2]:
# 주요 Feature 간 상관관계 분석
#
# Sleep_Duration이 Sleep_Efficiency와는 관계가 있지만
# Stress_Level, Daytime_Sleepiness와도 강한 관계를 가지고 있는지 확인

analysis_features = [
    "Sleep_Duration",
    "Daytime_Sleepiness",
    "Stress_Level",
    "Sleep_Efficiency"
]

correlation_matrix = data[
    analysis_features
].corr()

print("=== 주요 Feature 상관관계 ===")

display(
    correlation_matrix.round(4)
)

=== 주요 Feature 상관관계 ===


,Sleep_Duration,Daytime_Sleepiness,Stress_Level,Sleep_Efficiency
Sleep_Duration,1.0000,-0.5954,-0.7364,0.5611
Daytime_Sleepiness,-0.5954,1.0000,0.8117,-0.7814
Stress_Level,-0.7364,0.8117,1.0000,-0.7643
Sleep_Efficiency,0.5611,-0.7814,-0.7643,1.0000


상관관계 분석 결과

Sleep_Duration은 Sleep_Efficiency와 0.5611의 양의 상관관계를 보임.

그러나 Sleep_Duration은
Stress_Level과 -0.7364,
Daytime_Sleepiness와 -0.5954의 상관관계를 보임.

또한 Stress_Level과 Daytime_Sleepiness 사이에서도
0.8117의 강한 양의 상관관계가 확인됨.

따라서 주요 입력 Feature 사이에 정보가 중복되어
다중회귀에서 Sleep_Duration의 독립적인 영향력이
작아졌을 가능성이 있음.

다음 단계에서 VIF를 계산하여 다중공선성을 추가 확인함.

In [3]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor


vif_features = [
    "Sleep_Duration",
    "Daytime_Sleepiness",
    "Stress_Level"
]

X_vif = data[vif_features].copy()

# Linear Regression의 절편과 동일하게 상수항 추가
X_vif = sm.add_constant(X_vif)

vif_result = pd.DataFrame({
    "Feature": X_vif.columns,
    "VIF": [
        variance_inflation_factor(X_vif.values, i)
        for i in range(X_vif.shape[1])
    ]
})

# const는 Feature가 아니므로 제외
vif_result = vif_result[
    vif_result["Feature"] != "const"
]

print("=== 주요 Feature VIF ===")
display(vif_result.sort_values("VIF", ascending=False).round(4))

=== 주요 Feature VIF ===


,Feature,VIF
3,Stress_Level,4.1341
2,Daytime_Sleepiness,2.9313
1,Sleep_Duration,2.1848


VIF 분석 결과

상수항을 포함하여 VIF를 계산한 결과
Stress_Level       : 4.1341
Daytime_Sleepiness : 2.9313
Sleep_Duration     : 2.1848

모든 Feature의 VIF가 5 미만으로 나타났으므로
심각한 다중공선성이 존재한다고 보기는 어려움.

주요 Feature 사이에 상관관계와 일부 정보 중복은 존재하지만,
이것만으로 Sleep_Duration의 회귀계수가 매우 작아진 현상을
완전히 설명하기는 어려움.

따라서 다음 단계에서는 Feature를 하나씩 제외한 모델을 학습하여
 Sleep_Duration의 계수와 모델 성능 변화를 비교함.

모델 A
Sleep_Duration만 사용

모델 B
Sleep_Duration + Daytime_Sleepiness

모델 C
Sleep_Duration + Daytime_Sleepiness + Stress_Level

In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split


# 비교할 Feature 조합
feature_sets = {
    "A_Sleep_Duration": [
        "Sleep_Duration"
    ],

    "B_+Daytime_Sleepiness": [
        "Sleep_Duration",
        "Daytime_Sleepiness"
    ],

    "C_+Stress_Level": [
        "Sleep_Duration",
        "Daytime_Sleepiness",
        "Stress_Level"
    ]
}


results = []

for model_name, features in feature_sets.items():

    X = data[features]
    y = data["Sleep_Efficiency"]

    # 동일한 조건으로 Train/Test 분리
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    # Linear Regression 학습
    model = LinearRegression()
    model.fit(X_train, y_train)

    # Test 예측
    y_pred = model.predict(X_test)

    # Sleep_Duration 계수
    sleep_duration_coef = model.coef_[
        features.index("Sleep_Duration")
    ]

    results.append({
        "Model": model_name,
        "Features": len(features),
        "Sleep_Duration_Coefficient": sleep_duration_coef,
        "Test_MAE": mean_absolute_error(y_test, y_pred),
        "Test_R2": r2_score(y_test, y_pred)
    })


results_df = pd.DataFrame(results)

print("=== Feature 추가에 따른 Sleep Duration 영향 변화 ===")
display(results_df.round(4))

=== Feature 추가에 따른 Sleep Duration 영향 변화 ===


,Model,Features,Sleep_Duration_Coefficient,Test_MAE,Test_R2
0,A_Sleep_Duration,1,3.6690,5.1340,0.3157
1,B_+Daytime_Sleepiness,2,0.9785,3.8085,0.6237
2,C_+Stress_Level,3,0.0089,3.6226,0.6629


Feature 추가에 따른 Sleep Duration 영향 분석

Sleep_Duration만 사용했을 때 회귀계수는 3.6690이었으나,
Daytime_Sleepiness를 추가하면 0.9785,
Stress_Level까지 추가하면 0.0089로 크게 감소함.

반면 Test R²는 0.3157 → 0.6237 → 0.6629로 증가함.

따라서 Sleep_Duration 자체가 Sleep_Efficiency와 관계가 없는 것은 아니지만,
Daytime_Sleepiness와 Stress_Level을 함께 고려하면
Sleep_Duration이 추가적으로 제공하는 설명력이 매우 작아지는 것으로 확인됨.

이로 인해 최종 Linear Regression 모델에서는
Sleep_Duration만 변경했을 때 예측값 변화가 매우 작게 나타남.

In [5]:
# 전체 데이터에서 Sleep_Duration과 Sleep_Efficiency 상관관계
overall_corr = data[
    ["Sleep_Duration", "Sleep_Efficiency"]
].corr().iloc[0, 1]

print("=== 전체 데이터 ===")
print(
    f"Sleep_Duration ↔ Sleep_Efficiency: "
    f"{overall_corr:.4f}"
)


# Stress_Level별 상관관계
stress_corr = (
    data.groupby("Stress_Level")
    .apply(
        lambda x: x["Sleep_Duration"].corr(
            x["Sleep_Efficiency"]
        ),
        include_groups=False
    )
)

print("\n=== Stress Level별 상관관계 ===")
display(
    stress_corr.rename("Correlation").round(4)
)

=== 전체 데이터 ===
Sleep_Duration ↔ Sleep_Efficiency: 0.5611

=== Stress Level별 상관관계 ===


Stress_Level
4   -0.0055
7    0.0060
9   -0.0202
Name: Correlation, dtype: float64

In [6]:
# Stress_Level과 Daytime_Sleepiness가 모두 같은 데이터끼리 묶어서
# Sleep_Duration과 Sleep_Efficiency의 상관관계 확인

conditional_corr = (
    data.groupby(
        [
            "Stress_Level",
            "Daytime_Sleepiness"
        ]
    )
    .apply(
        lambda x: pd.Series({
            "Count": len(x),
            "Correlation": x["Sleep_Duration"].corr(
                x["Sleep_Efficiency"]
            )
        }),
        include_groups=False
    )
    .reset_index()
)


# 표본이 너무 적은 그룹 제외
conditional_corr = conditional_corr[
    conditional_corr["Count"] >= 30
]


print(
    "=== Stress Level + Daytime Sleepiness 통제 후 "
    "상관관계 ==="
)

display(
    conditional_corr.round(4)
)

=== Stress Level + Daytime Sleepiness 통제 후 상관관계 ===


c:\Users\EZ\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_function_base_impl.py:3028: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
c:\Users\EZ\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_function_base_impl.py:2901: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
c:\Users\EZ\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_function_base_impl.py:2901: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


,Stress_Level,Daytime_Sleepiness,Count,Correlation
0,4,1,4306.0,-0.0099
1,4,2,3320.0,0.0053
2,4,3,2587.0,0.0190
3,4,4,1295.0,-0.0188
4,4,5,411.0,0.0269
5,4,6,86.0,-0.0926
9,7,1,81.0,-0.0323
10,7,2,417.0,0.0787
11,7,3,1169.0,0.0950
12,7,4,2462.0,0.0003
